# WhisperX 語音轉文字（Google Colab）

功能：
- 語音 / 影片自動轉文字（繁中、英文、日文…）
- **逐字時間戳**（word-level timestamps）
- **說話者識別**（speaker diarization）
- 輸出 SRT / VTT / TXT / JSON

> **使用前請確認：執行階段 → 變更執行階段類型 → GPU（T4）**

## 1. 安裝套件

In [ ]:
# 確認 GPU
!nvidia-smi

# 安裝 WhisperX（含相依套件）
!pip install -q whisperx

# 若需要說話者識別，另需安裝 pyannote
!pip install -q pyannote.audio

## 2. 設定參數

In [ ]:
# ── 基本設定 ──────────────────────────────────────────────
AUDIO_FILE    = "/content/audio.mp3"   # 音訊 / 影片檔路徑
MODEL_NAME    = "large-v2"             # tiny / base / small / medium / large-v2 / large-v3
LANGUAGE      = "zh"                   # 'zh'=中文, 'en'=英文, None=自動偵測
DEVICE        = "cuda"                 # 'cuda' or 'cpu'
COMPUTE_TYPE  = "float16"              # float16 / int8 / float32
BATCH_SIZE    = 16                     # GPU 記憶體不足時調低

# ── 說話者識別（可選）────────────────────────────────────
DIARIZE       = False                  # True 開啟說話者識別
HF_TOKEN      = ""                    # HuggingFace Access Token（diarize=True 時必填）
MIN_SPEAKERS  = None                   # 最少說話者人數（None=自動）
MAX_SPEAKERS  = None                   # 最多說話者人數（None=自動）

# ── 輸出設定 ─────────────────────────────────────────────
OUTPUT_DIR    = "/content/output"      # 輸出資料夾
OUTPUT_FORMAT = "all"                  # srt / vtt / txt / json / all

## 3. 上傳音訊檔

可選擇以下任一方式：
- **本機上傳**：執行下一個 cell
- **Google Drive**：掛載後直接填入路徑
- **URL 下載**：使用 `!wget -O /content/audio.mp3 "<URL>"`

In [ ]:
# 方式 A：從本機上傳
from google.colab import files
uploaded = files.upload()
AUDIO_FILE = "/content/" + list(uploaded.keys())[0]
print(f"上傳完成：{AUDIO_FILE}")

In [ ]:
# 方式 B：掛載 Google Drive
# from google.colab import drive
# drive.mount('/content/drive')
# AUDIO_FILE = "/content/drive/MyDrive/your_audio.mp3"

## 4. 執行轉錄

In [ ]:
import whisperx
import os, json

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── Step 1: 載入模型 & 轉錄 ──────────────────────────────
print(f"載入模型：{MODEL_NAME}")
model = whisperx.load_model(MODEL_NAME, DEVICE, compute_type=COMPUTE_TYPE)

print("讀取音訊…")
audio = whisperx.load_audio(AUDIO_FILE)

print("轉錄中…")
result = model.transcribe(audio, batch_size=BATCH_SIZE, language=LANGUAGE)
print(f"偵測語言：{result['language']}")
print(f"片段數：{len(result['segments'])}")

In [ ]:
# ── Step 2: 對齊（逐字時間戳）────────────────────────────
print("對齊時間戳…")
align_model, metadata = whisperx.load_align_model(
    language_code=result["language"], device=DEVICE
)
result = whisperx.align(
    result["segments"], align_model, metadata, audio, DEVICE,
    return_char_alignments=False
)
print("對齊完成")

In [ ]:
# ── Step 3: 說話者識別（可選）────────────────────────────
if DIARIZE and HF_TOKEN:
    print("說話者識別中…")
    diarize_model = whisperx.DiarizationPipeline(
        use_auth_token=HF_TOKEN, device=DEVICE
    )
    diarize_kwargs = {}
    if MIN_SPEAKERS: diarize_kwargs["min_speakers"] = MIN_SPEAKERS
    if MAX_SPEAKERS: diarize_kwargs["max_speakers"] = MAX_SPEAKERS
    diarize_segments = diarize_model(audio, **diarize_kwargs)
    result = whisperx.assign_word_speakers(diarize_segments, result)
    print("說話者識別完成")
elif DIARIZE:
    print("警告：請設定 HF_TOKEN 以使用說話者識別")

## 5. 輸出結果

In [ ]:
from pathlib import Path

stem = Path(AUDIO_FILE).stem
segments = result.get("segments", [])

def fmt_ts(sec, style="srt"):
    h, rem = divmod(int(sec), 3600)
    m, s = divmod(rem, 60)
    ms = int((sec - int(sec)) * 1000)
    sep = "," if style == "srt" else "."
    return f"{h:02d}:{m:02d}:{s:02d}{sep}{ms:03d}"

formats = ["srt", "vtt", "txt", "json"] if OUTPUT_FORMAT == "all" else [OUTPUT_FORMAT]

for fmt in formats:
    out = Path(OUTPUT_DIR) / f"{stem}.{fmt}"
    with open(out, "w", encoding="utf-8") as f:
        if fmt == "txt":
            for seg in segments:
                sp = f"[{seg['speaker']}] " if "speaker" in seg else ""
                f.write(f"{sp}{seg['text'].strip()}\n")
        elif fmt == "srt":
            for i, seg in enumerate(segments, 1):
                sp = f"[{seg['speaker']}] " if "speaker" in seg else ""
                f.write(f"{i}\n{fmt_ts(seg['start'])} --> {fmt_ts(seg['end'])}\n{sp}{seg['text'].strip()}\n\n")
        elif fmt == "vtt":
            f.write("WEBVTT\n\n")
            for i, seg in enumerate(segments, 1):
                sp = f"<v {seg['speaker']}>" if "speaker" in seg else ""
                f.write(f"{i}\n{fmt_ts(seg['start'],'vtt')} --> {fmt_ts(seg['end'],'vtt')}\n{sp}{seg['text'].strip()}\n\n")
        elif fmt == "json":
            json.dump(result, f, ensure_ascii=False, indent=2)
    print(f"已儲存：{out}")

## 6. 預覽結果

In [ ]:
# 顯示前 10 個片段
for seg in segments[:10]:
    speaker = seg.get('speaker', '')
    sp_label = f"[{speaker}] " if speaker else ""
    print(f"{fmt_ts(seg['start'])} → {fmt_ts(seg['end'])}  {sp_label}{seg['text'].strip()}")

In [ ]:
# 下載輸出檔案
from google.colab import files
import glob

for f in glob.glob(f"{OUTPUT_DIR}/{stem}.*"):
    files.download(f)